# Simulation Setup

This page shows how to go from **built structures** (Frames/Atomistics) to
**ready‑to‑run simulation input** for external engines such as LAMMPS.

The goal is to keep the pipeline:

> *Model → Frame + Box + Force field → Engine‑specific input files*

---

## 1. From molecular model to Frame

No matter how you build your system—SMILES/BigSMILES + builder, reaction workflows,
manual scripting—the end result should be:

- A `Frame` with `"atoms"` (and optionally `"bonds"`, `"angles"`, …)
- A `Box` stored in `frame.metadata["box"]`
- A force‑field object compatible with your target engine

If you start from `Atomistic`, you typically:

1. Build an `Atomistic` structure and assign types/charges with a typifier.
2. Convert to `Frame` (and `Box`) using IO or adapter helpers.

Refer to:

- `user-guide/molecular-building.md`
- `user-guide/reaction-modeling.md`
- `user-guide/data-structures.md`

for the modeling side.

---

## 2. Writing a complete LAMMPS system

MolPy provides a convenience writer to dump both structure and force field
into a directory that LAMMPS can use.



In [ ]:
import molpy as mp
from molpy.io import write_lammps_system

# Assume you already have:
# - frame: Frame with atoms, bonds, and a Box in metadata["box"]
# - ff:    ForceField object compatible with the system

write_lammps_system("lammps_system", frame, ff)



This will create something like:

```text
lammps_system/
├─ lammps_system.data   # structure (atoms, bonds, box, …)
└─ lammps_system.ff     # force‑field parameters
```

You can then point your LAMMPS input script at these files.

If you only need the data file, use `write_lammps_data` directly
(see `user-guide/file-io.md`).

---

## 3. Running LAMMPS via the engine interface

MolPy’s `engine` module wraps external executables in a simple Python API.
For LAMMPS there is `LAMMPSEngine`, which works with `Script` objects that
represent input scripts.



In [ ]:
from molpy.core.script import Script
from molpy.engine import LAMMPSEngine

# 1. Create a LAMMPS input script
script = Script.from_text(
    name="in.lammps",
    text=\"\"\"units real
atom_style full
read_data lammps_system.data
# ... rest of your LAMMPS commands ...
\"\"\",
    language="other",
)

# 2. Prepare engine
engine = LAMMPSEngine(executable="lmp")     # or your LAMMPS command
engine.prepare(work_dir="./run_lammps", scripts=[script])

# 3. Run
result = engine.run()
print("LAMMPS return code:", result.returncode)
print("stdout snippet:", result.stdout.splitlines()[:5])



This pattern keeps:

- The **structure & force field** handled by `Frame` + `write_lammps_system`
- The **MD run** handled by an `Engine` + `Script`

You can apply the same pattern to other engines once engine wrappers exist.

---

## 4. Putting it all together

A typical end‑to‑end workflow looks like:

1. **Modeling**
   - Build monomers/assemblies with `Atomistic` + wrappers (`Monomer`, `Polymer`).
   - Use `reacter` to define and apply chemistry where needed.
   - Optionally pack many copies using `molpy.pack`.

2. **Convert to simulation data**
   - Convert to a `Frame` with `"atoms"`/`"bonds"` and a `Box`.
   - Attach a compatible `ForceField` object.

3. **Export input files**
   - Use `write_lammps_system` (and friends) to generate `.data` + `.ff`.
   - Or use lower‑level writers as needed.

4. **Run the engine**
   - Create engine input scripts with `Script`.
   - Use `LAMMPSEngine` or other `Engine` subclasses to run calculations.

By keeping all steps expressed in terms of a few core types (`Frame`, `Box`,
`Atomistic`, `ForceField`, `Script`, `Engine`), MolPy aims to make simulation
setup both **transparent** and **easy to automate**.

